# 01 — The Phase 1 catalog, explored

Doc 11 §5 rule 1: **notebooks are never imported from.** Everything called
below lives in `src/hydrocascade/`; this notebook only looks at it. If you
find yourself writing production logic in a cell, move it into `src/` and
call it from here instead.

The point of this notebook is the part a report cannot do: let you poke at
a single reach, a single year, a single threshold, and see the detection
rule behave. The fixed conclusions live in:

- `reports/phase1_flood_definition.md` — the rule and its sensitivity
- `reports/quality_gates.md` — Doc 04 §6 and Doc 05 §9
- `LIMITATIONS.md` — read before quoting anything

**Prerequisite:** run the pipeline first (see the README). This notebook
reads artefacts, it does not build them.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hydrocascade import paths
from hydrocascade.config import load_config
from hydrocascade.catalog import schema, completeness, dedup
from hydrocascade.ingest import floods, adapters, corroboration

cfg = load_config()
pd.set_option('display.width', 160)

synthetic = adapters.synthetic_sources(cfg)
if synthetic:
    print('SYNTHETIC STAND-INS ACTIVE:', ', '.join(synthetic))
    print('Nothing derived from these is a result. See LIMITATIONS.md §2.')
cfg.region.id, cfg.period.start, cfg.period.end

## 1. What is in the catalog

One row per event, the 21 columns of Doc 04 §2, in that order.

In [ ]:
events = schema.read(paths.CATALOG / 'events.csv')
print(f'{len(events):,} events')
print(events.groupby(['hazard_type', 'source']).size())
events.head(3).T

In [ ]:
# Doc 04 §2 rule: every row carries a source_url and an accessed_by.
# `validate` returns a list of problems rather than raising, so you can
# see all of them at once instead of fixing them one exception at a time.
problems = schema.validate(events, strict=False)
print(f'{len(problems)} schema problem(s)')
for p in problems[:20]:
    print(' -', p)

### The rows that are deliberately *not* here

Doc 04 §5 names six documented events. Five have no sourced coordinates,
and Doc 04 §2 admits no exceptions to the source-URL rule — so they live in
`events_PENDING.csv` rather than being given invented coordinates to make
`events.csv` look fuller. This is the file to work through first.

In [ ]:
pending = pd.read_csv(paths.CATALOG / 'events_PENDING.csv', dtype=str,
                      keep_default_na=False)
pending[['date', 'lat', 'lon', 'source', 'notes']]

## 2. Doc 04 §6, gate by gate

`hydrocascade check` runs these non-interactively. Here you can see what
each one is actually looking at.

In [ ]:
land = events[events.hazard_type == 'L']
real = land[land.source != 'SYNTHETIC']
print(f'gate 1  volume: {len(land):,} landslide rows, of which '
      f'{len(real):,} are real')
print('        -> BLOCKED while synthetic rows are present: Doc 04 §6 '
      'gate 1 counts real events only.')
if len(real) < 200:
    print('        -> Doc 04 §6 HARD STOP applies if this is still true at '
          'the end of week 3:')
    print('           widen the region to the full Western Ghats before '
          'doing anything else.')

In [ ]:
# gate 3: seasonality. If events do not follow the monsoon, suspect dates
# or timezones before suspecting geology.
month = pd.to_datetime(land.date, errors='coerce').dt.month
counts = month.value_counts().reindex(range(1, 13), fill_value=0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(counts.index, counts.values,
       color=['#c7c7c7' if m not in (6, 7, 8, 9) else '#2b6ea3'
              for m in counts.index])
ax.set_xticks(range(1, 13))
ax.set_xlabel('month'); ax.set_ylabel('landslides')
ax.set_title(f'{100 * month.between(6, 9).mean():.0f}% in Jun-Sep '
             f'(south-west monsoon)')
plt.tight_layout()

### Gate 4 — the completeness cutoff

Doc 04 §3.4. Real landslide inventories follow an inverse power law above a
rollover and are **systematically incomplete below it**. Fitting a point
process on the incomplete part without saying so produces biased background
rates, and this one plot pre-empts the objection.

`alpha` is the Clauset–Shalizi–Newman MLE exponent; `A_min` is the modal bin
of the log-binned frequency density.

In [ ]:
areas = pd.to_numeric(land.area_m2, errors='coerce').dropna().to_numpy()
comp = completeness.find_rollover(areas, n_bins=int(cfg.catalog.completeness.n_bins))
print(comp.summary())

ok = comp.density > 0
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(comp.bin_centres[ok], comp.density[ok], 'o', ms=5, color='#2b6ea3')
ax.axvline(comp.a_min_m2, color='#d1341f', ls='--',
           label=f'A_min = {comp.a_min_m2:,.0f} m2')
ax.set_xlabel('landslide area (m2)')
ax.set_ylabel('frequency density dN/dA')
ax.legend(); ax.set_title(f'alpha = {comp.alpha:.2f} +/- {comp.alpha_stderr:.2f}')
plt.tight_layout()

In [ ]:
# Setting catalog.completeness.min_area_m2 in the config applies the cutoff
# on the next `hydrocascade catalog`. It is deliberately NOT applied
# automatically: dropping events is a decision, not a default.
kept = completeness.apply_cutoff(land, comp.a_min_m2)
print(f'{len(kept):,} of {len(land):,} landslides survive a '
      f'{comp.a_min_m2:,.0f} m2 cutoff')

## 3. The flood rule, one reach at a time

This is the cell the report cannot replace. Pick a reach, look at its
hydrograph, and watch the Q2 threshold and the 3-day declustering decide
what is one flood and what is two.

In [ ]:
from hydrocascade.spatial.network import Network

net = Network()
reaches = net.gdf[['reach_id', 'area_dn_km2']].copy()
areas_km2 = reaches.set_index('reach_id')['area_dn_km2']
disch = adapters.discharge(cfg, reaches)
print(f'discharge table: {disch.shape[0]:,} days x {disch.shape[1]:,} reaches')
print('synthetic:', disch.attrs.get('synthetic', False))

In [ ]:
# Change this to explore. Big catchments give smoother, later hydrographs.
REACH = int(areas_km2.sort_values().index[-1])   # the largest catchment
YEAR = 2018

q = disch[REACH]
amax = q.groupby(q.index.year).max().to_numpy()
thr = floods.return_period_thresholds(amax, tuple(cfg.floods.return_periods))
print(f'R-{REACH:05d}  {areas_km2[REACH]:,.0f} km2   ' +
      '  '.join(f'{k}={v:,.0f}' for k, v in thr.items()))

sub = q[q.index.year == YEAR]
evs = floods.detect_events(sub.to_numpy(), thr['Q2'],
                           int(cfg.floods.decluster_days))
print(f'{len(evs)} events in {YEAR}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sub.index, sub.values, lw=0.8, color='#3b7ea1')
for name, colour in (('Q2', '#d1341f'), ('Q5', '#e08214'), ('Q20', '#7b3294')):
    ax.axhline(thr[name], color=colour, lw=1, ls='--', label=name)
for e in evs:
    ax.axvspan(sub.index[e.start_index], sub.index[e.end_index],
               color='#d1341f', alpha=0.15)
ax.set_yscale('log'); ax.legend(loc='upper left')
ax.set_ylabel('discharge (m3/s)')
ax.set_title(f'R-{REACH:05d}, {YEAR}: shaded spans are detected events')
plt.tight_layout()

**Things worth trying here**

- Set `decluster_days` to 1 and watch a single week-long flood split into
  five events. That is the failure Doc 02 §3.1 step 3 exists to prevent, and
  it corrupts exactly the self-excitation the model is supposed to measure.
- Detect on `thr['Q5']` instead and count how many events survive.
- Pick a small headwater reach and see why the 50 km² eligibility filter
  exists.

The exceedance rate is the first diagnostic to read, and the intuitive
expectation is wrong: a Q_T threshold implies `-ln(1 - 1/T)` events per
reach-year, which is **0.69** for Q2 and **0.22** for Q5, not 1.0 for both.

In [ ]:
for t in (2, 5, 20):
    rate = floods.expected_exceedance_rate(t)
    print(f'Q{t:<3} implies {rate:.3f} events per reach-year')

## 4. The blind spot

Doc 02 §3.4 says the discharge rule cannot see the flood at the debris-flow
site. We have the catchment, so it is checkable rather than quotable.

In [ ]:
blind = corroboration.blind_spot(cfg)
print(f"injection reach   R-{blind['injection_reach']:05d}  "
      f"{blind['injection_area_km2']:.1f} km2")
print(f"Chooralmala       R-{blind['chooralmala_reach']:05d}  "
      f"{blind['chooralmala_area_km2']:.1f} km2")
for thr_km2, d in blind['thresholds'].items():
    if d['visible']:
        print(f"  at {thr_km2:>5.0f} km2: blind for {d['reaches_blind']} reaches "
              f"({d['hours_from_injection']:.1f} h), then R-{d['first_reach']:05d}")
    else:
        print(f"  at {thr_km2:>5.0f} km2: the whole chain is invisible")

## 5. What to do next in this notebook

In rough order of how much it unblocks:

1. **Get a real landslide inventory in.** Drop an ESSD / NRSC / Bhukosh
   download into `data/raw/<SOURCE>/<date>/` with a `PROVENANCE.md` and
   re-run `hydrocascade catalog`. The readers already accept CSV, SHP and
   GPKG with the usual column synonyms, and convert polygons to crown
   points (Doc 04 §3.2 — highest elevation, **not** centroid).
2. **Register for a Copernicus CDS account** and switch
   `data_mode.glofas: real`. Everything in section 3 above becomes a result.
3. **Re-source `data/reference/documented_floods.csv`** to primary KSDMA /
   Copernicus EMS / CWC records. Until then the recall number is circular:
   its denominator comes from the project's own brief.
4. **Resolve the Puthumala date** (8 vs 29 August 2019) from GSI or the
   primary field report, and record which source you trusted. Doc 04 §5
   calls this the team's training exercise in provenance discipline.